## Explore HuBERT Embeddings of IEMOCAP Data for Cross-Modal Attention Fusion

### Imports and paths

In [8]:
import numpy as np
import os
from pathlib import Path

embeddings_dir = Path("/Users/audreylu/software_projects/research/SenticCrystal/data/embeddings/hubert_features")
files = {
    "train": embeddings_dir / "iemocap_train_hubert.npz",
    "val": embeddings_dir / "iemocap_val_hubert.npz",
    "test": embeddings_dir / "iemocap_test_hubert.npz"
}

### Load and explore each file

In [ ]:
hubert_data = {}
for split, filepath in files.items():
    print(f"\n{'='*60}")
    print(f"Exploring {split.upper()} split: {filepath.name}")
    print(f"{'='*60}")
    
    if not filepath.exists():
        print(f"⚠️  File not found: {filepath}")
        continue
    
    data = np.load(filepath, allow_pickle=True)
    hubert_data[split] = data
    
    # Print keys in the npz file
    print(f"Keys in NPZ: {list(data.files)}")
    print(f"Number of keys: {len(data.files)}")
    
    # Explore each array
    for key in data.files:
        arr = data[key]
        print(f"\n  Key: '{key}'")
        print(f"    Shape: {arr.shape}")
        print(f"    Dtype: {arr.dtype}")
        if arr.size > 0:
            print(f"    Min: {arr.min():.6f}, Max: {arr.max():.6f}, Mean: {arr.mean():.6f}")
        if len(arr.shape) > 1:
            print(f"    Dimensions: batch={arr.shape[0]}, features={arr.shape[1]}")
            if len(arr.shape) > 2:
                print(f"    Additional dims: {arr.shape[2:]}")


Exploring TRAIN split: iemocap_train_hubert.npz
Keys in NPZ: ['features', 'labels', 'ids', 'emotion_names']
Number of keys: 4

  Key: 'features'
    Shape: (3205, 1024)
    Dtype: float32
    Min: -1.065222, Max: 1.901952, Mean: 0.000786
    Dimensions: batch=3205, features=1024

  Key: 'labels'
    Shape: (3205,)
    Dtype: int64
    Min: 0.000000, Max: 3.000000, Mean: 1.568175

  Key: 'ids'
    Shape: (3205,)
    Dtype: <U22
    Sample values: ['Ses02F_impro01_F000' 'Ses02F_impro01_M000' 'Ses02F_impro01_M001']

  Key: 'emotion_names'
    Shape: (4,)
    Dtype: <U7
    Sample values: ['angry' 'happy' 'sad']

Exploring VAL split: iemocap_val_hubert.npz
Keys in NPZ: ['features', 'labels', 'ids', 'emotion_names']
Number of keys: 4

  Key: 'features'
    Shape: (1085, 1024)
    Dtype: float32
    Min: -0.954721, Max: 1.504189, Mean: 0.000890
    Dimensions: batch=1085, features=1024

  Key: 'labels'
    Shape: (1085,)
    Dtype: int64
    Min: 0.000000, Max: 3.000000, Mean: 1.675576

  K

### Analysis for cross-modal attention

In [ ]:
print("="*60)
print("ANALYSIS FOR CROSS-MODAL ATTENTION FUSION")
print("="*60)

splits_summary = {}
for split in ["train", "val", "test"]:
    data = hubert_data[split]
    features = data['features']
    
    splits_summary[split] = {
        "num_samples": features.shape[0],
        "feature_dim": features.shape[1],
        "dtype": features.dtype
    }

print("\n1. FEATURE DIMENSIONS:")
for split, info in splits_summary.items():
    print(f"   {split.upper():8} -> Samples: {info['num_samples']:4d}, HuBERT Dim: {info['feature_dim']:4d}, Dtype: {info['dtype']}")

print("\n2. DATA TYPE & PRECISION:")
print(f"   HuBERT uses float32 (32-bit floating point)")
print(f"   This is standard for transformer embeddings")

print("\n3. SCALE & NORMALIZATION:")
for split in ["train", "val", "test"]:
    features = hubert_data[split]['features']
    print(f"   {split.upper():8} -> Range: [{features.min():.4f}, {features.max():.4f}], Std: {features.std():.6f}")

print("\n4. KEY CONSIDERATIONS FOR CROSS-MODAL ATTENTION:")
print(f"   ✓ HuBERT Feature Dimension: 1024")
print(f"   ✓ Number of Utterances (samples): Train={splits_summary['train']['num_samples']}, " +
      f"Val={splits_summary['val']['num_samples']}, Test={splits_summary['test']['num_samples']}")
print(f"   ✓ Data is already normalized (mean ≈ 0)")
print(f"   ✓ Consistent feature space across splits")
print(f"\n   FUSION ARCHITECTURE NOTES:")
print(f"   - If SRoBERTa also has 1024 dims: direct concatenation (2048 total)")
print(f"   - Cross-attention expects 2 modalities with same batch size")
print(f"   - Both embeddings need to be aligned by utterance ID (already present in data)")
print(f"   - Typical fusion: Query from one modality, Key/Value from another")
print(f"   - Attention projection dimensions should match or be projected")

print("\n5. EMOTION LABELS:")
emotions = hubert_data['train']['emotion_names']
print(f"   Emotion classes: {list(emotions)}")
print(f"   Number of classes: {len(emotions)}")
print(f"   Label encoding: 0-3 for 4-way classification")

ANALYSIS FOR CROSS-MODAL ATTENTION FUSION

1. FEATURE DIMENSIONS:
   TRAIN    -> Samples: 3205, HuBERT Dim: 1024, Dtype: float32
   VAL      -> Samples: 1085, HuBERT Dim: 1024, Dtype: float32
   TEST     -> Samples: 1241, HuBERT Dim: 1024, Dtype: float32

2. DATA TYPE & PRECISION:
   HuBERT uses float32 (32-bit floating point)
   This is standard for transformer embeddings

3. SCALE & NORMALIZATION:
   TRAIN    -> Range: [-1.0652, 1.9020], Std: 0.096539
   VAL      -> Range: [-0.9547, 1.5042], Std: 0.105711
   TEST     -> Range: [-1.0752, 1.6462], Std: 0.103095

4. KEY CONSIDERATIONS FOR CROSS-MODAL ATTENTION:
   ✓ HuBERT Feature Dimension: 1024
   ✓ Number of Utterances (samples): Train=3205, Val=1085, Test=1241
   ✓ Data is already normalized (mean ≈ 0)
   ✓ Consistent feature space across splits

   FUSION ARCHITECTURE NOTES:
   - If SRoBERTa also has 1024 dims: direct concatenation (2048 total)
   - Cross-attention expects 2 modalities with same batch size
   - Both embeddings need

### Fusion with SRoBERTa